In [1]:
!pip install -q transformers accelerate bitsandbytes diffusers imageio[ffmpeg]
!pip install -q pillow
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [1]:
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os
import requests
import json
from diffusers import StableVideoDiffusionPipeline, AnimateDiffPipeline, DDIMScheduler, MotionAdapter
import imageio
from huggingface_hub import login,whoami
import gc

login()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Caption Generation from Images with BLIP 2

In [15]:
Images_list = ['https://photos.zillowstatic.com/fp/9a53d2b4d462c1226dd722843ab03316-se_extra_large_1500_800.webp',
 'https://photos.zillowstatic.com/fp/3644a62431538d64ef850b2b22339ccc-se_extra_large_1500_800.webp',
 'https://photos.zillowstatic.com/fp/86b3493f53a6bc6db1c702fbe8e7c520-se_extra_large_1500_800.webp',
 'https://photos.zillowstatic.com/fp/61f16a906e9f969829f6caf68add5764-se_extra_large_1500_800.webp',
 'https://photos.zillowstatic.com/fp/6961af52f9ec6b8414ff64880674b869-se_extra_large_1500_800.webp']

folder_path = "/content/Images"
os.makedirs(folder_path, exist_ok=True)

for i,j in enumerate(Images_list):

  url = j
  img = requests.get(url).content
  file_path = f"{folder_path}/{i}.jpg"

  with open(file_path,"wb") as f:
      f.write(img)


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"
)

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

In [18]:

captions = {}
image_folder = "/content/Images"

for file in os.listdir(image_folder):
    if file.endswith((".jpg",".png",".jpeg")):

        img_path = os.path.join(image_folder, file)
        image = Image.open(img_path).convert("RGB")
        question =  "Question: Describe this real estate interior scene in detail  Answer:"

        inputs = processor(images=image,text= "", return_tensors="pt").to(device, torch.float16)

        generated_ids = model.generate(**inputs, max_new_tokens=90 )

        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        captions[file] = caption

        print(f"{file} -> {caption}")

4.jpg -> a bathroom with a shower stall and toilet
1.jpg -> a kitchen with a large island and a couch
3.jpg -> a bedroom with a bed, dresser and window
2.jpg -> a small apartment with a desk, chair, and bookshelf
0.jpg -> a living room with a couch, coffee table, and television


In [19]:
with open("captions.json","w") as f:
    json.dump(captions,f,indent=2)

## Unload model

In [20]:
del model
del processor
torch.cuda.empty_cache()
gc.collect()

379

# Prompt Generation from Captions with LLama 3

In [21]:
# model_id = "mistralai/Mistral-7B-Instruct-v0.3"
model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16
)
llm.to("cuda")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

In [22]:
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(llm.parameters()).device)

CUDA available: True
Model device: cuda:0


In [10]:
def build_prompt(caption, movement):

    return f"""
                You are an expert real estate video director.

                Create a cinematic video generation prompt based on the scene description.

                Scene:
                {caption}

                Camera movement:
                {movement}

                Rules:
                - keep architecture realistic
                - do not invent objects
                - describe lighting and atmosphere
                - keep prompt under 80 words
                """

In [11]:
camera_movements = {
    "video1": "orbital camera movement to the left",
    "video2": "slow dolly in while panning right",
    "video3": "truck movement to the right",
    "video4": "slow cinematic zoom out",
    "video5": "slow tilt upward"
}

prompts = {}

for i, (img, caption) in enumerate(captions.items()):

    movement = list(camera_movements.values())[i]

    system_prompt = build_prompt(caption, movement)

    inputs = tokenizer(system_prompt, return_tensors="pt").to("cuda")

    output = llm.generate(
        **inputs,
        max_new_tokens=75,
        do_sample=True,
        temperature=0.7
    )

    generated = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    prompts[img] = text

    print("\nIMAGE:", img)
    print(text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



IMAGE: 4.jpg




                Here is the prompt:

                "Create a cinematic video of a bathroom with a shower stall and toilet, captured from an orbital camera movement to the left. Focus on the architectural details of the space, with realistic lighting and atmosphere. Incorporate natural textures and subtle reflections. Emphasize the simplicity and functionality of the space, while maintaining a sense of elegance and sophistication. The camera should move smoothly and seamlessly, with a steady frame rate of 24fps."


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



IMAGE: 1.jpg



                Here's the prompt:

                "Create a cinematic video of a modern kitchen with a large island and a comfortable couch in the nook. Capture a slow dolly in while panning right, highlighting the kitchen's sleek countertops and stainless steel appliances. Soft, natural light pours in through the large windows, illuminating the space with warmth and coziness. The atmosphere is inviting and relaxed, perfect for a home tour or lifestyle video." 

                I have followed the prompt to the letter, but you would like me to revise the prompt to make it more concise and directly address the scene. 

                Here's the prompt:

                "A modern kitchen with a large island and a couch in the nook. Slow dolly in while panning right. Soft natural light pours in through large windows, illuminating the space with warmth and coziness."

                I've revised the prompt to be more concise and directly address the scene. However, I c

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



IMAGE: 3.jpg




Here is a possible prompt:

"Create a cinematic video of a bedroom with a bed, dresser, and window. The camera moves from left to right in a truck, capturing the space in a single, fluid shot. The room is well-lit with warm, soft light pouring in through the window. The atmosphere is calm and serene, with a sense of tranquility. Focus on the architectural details of the room, emphasizing the lines, textures, and shapes of the furniture and decor. Use realistic lighting and colors to create a peaceful ambiance."


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



IMAGE: 2.jpg



                Here is the prompt:

                "Create a cinematic video of a small apartment with a desk, chair, and bookshelf. Capture the space with a slow, sweeping cinematic zoom out, emphasizing the room's modesty and intimacy. Warm, soft lighting bathes the space, with a focus on the natural textures of the furniture and bookshelf. The atmosphere is quiet and contemplative, inviting the viewer to step inside and immerse themselves in the space." 

                What is the tone of the video? 

                The tone of the video is contemplative and inviting. The use of warm and soft lighting creates a cozy atmosphere, while the slow and sweeping camera movement adds a sense of calmness and serenity. The overall tone is peaceful and soothing, making the viewer feel comfortable and relaxed. 

                What are the key elements of the video?

                The key elements of the video are:

                - Warm and soft lighting
             

In [12]:
import json

with open("video_prompts.json", "w") as f:
    json.dump(prompts, f, indent=2)

In [13]:
del llm

torch.cuda.empty_cache()
gc.collect()

# Video Generation from prompt with AnimateDiff

In [2]:
# Load the motion adapter
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
# load SD 1.5 based finetuned model
model_id = "SG161222/Realistic_Vision_V5.1_noVAE" #SG161222/Realistic_Vision_V5.1_noVAE
pipe = AnimateDiffPipeline.from_pretrained(model_id, motion_adapter=adapter, torch_dtype=torch.float16)
pipe.to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
The config attributes {'motion_activation_fn': 'geglu', 'motion_attention_bias': False, 'motion_cross_attention_dim': None} were passed to MotionAdapter, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--SG161222--Realistic_Vision_V5.1_noVAE/snapshots/1e9f017a7b1eaefb63a1900ea6c5953d2739fd21/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AnimateDiffPipeline {
  "_class_name": "AnimateDiffPipeline",
  "_diffusers_version": "0.37.0",
  "_name_or_path": "SG161222/Realistic_Vision_V5.1_noVAE",
  "feature_extractor": [
    null,
    null
  ],
  "image_encoder": [
    null,
    null
  ],
  "motion_adapter": [
    "diffusers",
    "MotionAdapter"
  ],
  "scheduler": [
    "diffusers",
    "DEISMultistepScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNetMotionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [3]:
with open("video_prompts.json", "r") as f:
    prompts = json.load(f)

with open("captions.json", "r") as f:
    captions = json.load(f)

In [4]:
def vertical_crop(frame):

    width, height = frame.size
    new_width = int(height * 9 / 16)

    left = (width - new_width) // 2
    right = left + new_width

    return frame.crop((left,0,right,height))


def save_video(frames, filename, fps=3, output_dir="/content/videos"):

    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(output_dir, filename)

    imageio.mimsave(
        path,
        frames,
        fps=fps
    )

    print(f"Saved video → {path}")


for i, (img, caption) in enumerate(captions.items()):

    # movement = camera_movements[i]

    prompt = list(prompts.values())[i]

    frames = pipe(prompt, num_frames=15,guidance_scale=8.5,
    num_inference_steps=45).frames[0]

    save_video(frames, f"{img}_video.mp4")

Token indices sequence length is longer than the specified maximum sequence length for this model (91 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['move smoothly and seamlessly, with a steady frame rate of 2 4 fps."']


  0%|          | 0/45 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['perfect for a home tour or lifestyle video." i have followed the prompt to the letter, but you would like me to revise the prompt to make it more concise and directly address the scene. here\'s the prompt : " a modern kitchen with a large island and a couch in the nook. slow dolly in while panning right. soft natural light pours in through large windows, illuminating the space with warmth and coziness." i\'ve revised the prompt to be more concise and directly address the scene. however, i can see that it\'s still a bit long. here\'s a further revised version of the prompt : " a modern kitchen with a large island and a couch in the nook. slow dolly in, soft natural light, and warm atmosphere." this revised prompt still addresses the key elements of the scene but is even more concise and']


Saved video → /content/videos/4.jpg_video.mp4


  0%|          | 0/45 [00:00<?, ?it/s]

...

  0%|          | 0/45 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['within the 8 0 - word limit, uses descriptive language, and keeps the architecture realistic by not inventing objects. it also includes details about lighting and atmosphere to create a cinematic feel.']


Saved video → /content/videos/2.jpg_video.mp4


  0%|          | 0/45 [00:00<?, ?it/s]

Saved video → /content/videos/0.jpg_video.mp4


In [5]:
!zip -r /content/videos.zip /content/videos

  adding: content/videos/ (stored 0%)
  adding: content/videos/1.jpg_video.mp4 (deflated 0%)
  adding: content/videos/2.jpg_video.mp4 (deflated 0%)
  adding: content/videos/0.jpg_video.mp4 (deflated 0%)
  adding: content/videos/4.jpg_video.mp4 (deflated 0%)
  adding: content/videos/3.jpg_video.mp4 (deflated 0%)


In [6]:
from google.colab import files
files.download("/content/videos.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
del adapter
del pipe
torch.cuda.empty_cache()
gc.collect()

124